# run_model_report
CPU per-model Stage 5 reporting worker. Run cells 2, 3 and 4 in order. Configure only cell 3. Select one `MODEL` (`qwen` or `gemma`) and supply the accepted Stage 4 release reference (`path` and `sha256`). Optionally set `RESUME_REF` to verify and reuse a completed compatible bundle published under the same code commit.


In [ ]:
import json, os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO = Path('/content/Research-stage5')
CODE_REF = 'phase1'  # Resolved once below; set this to the approved Stage 5 commit to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
MODEL = 'qwen'  # 'qwen' or 'gemma'
STAGE4_RELEASE = {'path': 'REPLACE_STAGE4_RELEASE/_COMPLETE.json', 'sha256': 'REPLACE_SHA256'}
RESUME_REF = None  # Optional {'path': '...', 'sha256': '...'} of a completed compatible report to reuse.


In [ ]:
if MODEL not in ('qwen', 'gemma'):
    raise RuntimeError(f"MODEL must be 'qwen' or 'gemma', got {MODEL!r}")
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Stage 5 checkout.')
if not isinstance(STAGE4_RELEASE, dict) or 'REPLACE_' in str(STAGE4_RELEASE.get('path', '')):
    raise RuntimeError('Set the explicit accepted STAGE4_RELEASE path and sha256 in cell 3.')
if not (PHASE1_ROOT / STAGE4_RELEASE['path']).is_file():
    raise RuntimeError(f"Stage 4 release manifest not found: {PHASE1_ROOT / STAGE4_RELEASE['path']}")
command = [
    sys.executable, '-u', '-m', 'src.stage5.run_model_report',
    '--phase1-root', str(PHASE1_ROOT),
    '--release', json.dumps(STAGE4_RELEASE),
    '--model', MODEL,
    '--storage-scope', 'persistent-drive',
]
if RESUME_REF is not None:
    command += ['--resume', json.dumps(RESUME_REF) if isinstance(RESUME_REF, dict) else str(RESUME_REF)]
process = subprocess.Popen(command, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait() != 0:
    raise RuntimeError(f'Stage 5 {MODEL} reporting worker blocked/failed.')
